In [ ]:
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
from shapely import wkt
import json
import warnings
warnings.filterwarnings('ignore')

# Load both datasets
nycha_locations = pd.read_csv("https://data.cityofnewyork.us/api/views/phvi-damg/rows.csv?accessType=DOWNLOAD")
nycha_databook = pd.read_csv("https://data.cityofnewyork.us/api/views/evjd-dqpz/rows.csv?accessType=DOWNLOAD")

# Convert geometry
nycha_locations['geometry'] = nycha_locations['the_geom'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(nycha_locations, geometry='geometry', crs='EPSG:4326')

# Clean and merge datasets
nycha_databook['DEVELOPMENT_CLEAN'] = nycha_databook['DEVELOPMENT'].str.strip().str.upper()
gdf['DEVELOPMEN_CLEAN'] = gdf['DEVELOPMEN'].str.strip().str.upper()

# Extract year from completion date
nycha_databook['COMPLETION_YEAR'] = pd.to_datetime(
    nycha_databook['COMPLETION DATE'], errors='coerce'
).dt.year

# Merge on development name
merged = gdf.merge(
    nycha_databook[['DEVELOPMENT_CLEAN', 'COMPLETION_YEAR', 'TOTAL POPULATION']],
    left_on='DEVELOPMEN_CLEAN',
    right_on='DEVELOPMENT_CLEAN',
    how='left'
)

# Remove rows without completion year
merged = merged[merged['COMPLETION_YEAR'].notna()].copy()

# Create decade categories
def assign_decade(year):
    if year < 1940:
        return 'Pre-1940'
    elif year < 1950:
        return '1940s'
    elif year < 1960:
        return '1950s'
    elif year < 1970:
        return '1960s'
    elif year < 1980:
        return '1970s'
    elif year < 1990:
        return '1980s'
    elif year < 2000:
        return '1990s'
    else:
        return '2000s+'

merged['DECADE'] = merged['COMPLETION_YEAR'].apply(assign_decade)

# Color scheme
decade_colors = {
    'Pre-1940': '#8B0000',
    '1940s': '#FF4500',
    '1950s': '#FFD700',
    '1960s': '#32CD32',
    '1970s': '#1E90FF',
    '1980s': '#9370DB',
    '1990s': '#FF1493',
    '2000s+': '#2F4F4F'
}

# Create base map
m = folium.Map(
    location=[40.7128, -73.95],
    zoom_start=11,
    tiles='cartodbpositron',
    control_scale=True
)

# Add title
title_html = '''
<div style="position: fixed; 
            top: 10px; 
            left: 50%; 
            transform: translateX(-50%);
            width: 600px; 
            background-color: white; 
            z-index: 9999; 
            font-size: 24px; 
            font-weight: bold;
            text-align: center;
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.2);">
    <div style="font-size: 28px; color: #1a1a1a; margin-bottom: 5px;">NYCHA Through the Decades</div>
    <div style="font-size: 14px; color: #666; font-weight: normal;">When New York City's Public Housing Was Built</div>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# Add data source attribution
attribution_html = '''
<div style="position: fixed; 
            bottom: 10px; 
            left: 10px; 
            background-color: white; 
            z-index: 9999; 
            font-size: 12px;
            padding: 8px 12px;
            border-radius: 4px;
            box-shadow: 0 1px 4px rgba(0,0,0,0.2);
            color: #666;">
    Data: NYC Open Data
</div>
'''
m.get_root().html.add_child(folium.Element(attribution_html))

# Load actual NYC Borough boundaries from NYC Open Data
try:
    boroughs_url = "https://data.cityofnewyork.us/api/geospatial/tqmj-j8zm?method=export&format=GeoJSON"
    boroughs_gdf = gpd.read_file(boroughs_url)
    
    # Add borough boundaries with proper styling
    folium.GeoJson(
        boroughs_gdf,
        name='Borough Boundaries',
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#333',
            'weight': 3,
            'dashArray': '5, 5',
            'opacity': 0.6
        }
    ).add_to(m)
    
    # Add borough labels at centroids
    for idx, row in boroughs_gdf.iterrows():
        centroid = row.geometry.centroid
        borough_name = row['boro_name']
        folium.Marker(
            [centroid.y, centroid.x],
            icon=folium.DivIcon(html=f'''
                <div style="font-size: 18px; 
                            font-weight: bold; 
                            color: #333; 
                            text-shadow: 2px 2px 4px white, -2px -2px 4px white, 
                                         2px -2px 4px white, -2px 2px 4px white;
                            white-space: nowrap;">
                    {borough_name}
                </div>
            ''')
        ).add_to(m)
    
except Exception as e:
    print(f"Could not load borough boundaries: {e}")
    print("Continuing without borough boundaries...")

# Count developments by decade
decade_counts = merged['DECADE'].value_counts().to_dict()

# Add NYCHA developments
for idx, row in merged.iterrows():
    # Get centroid for marker placement
    centroid = row.geometry.centroid
    
    # Population info
    if pd.notna(row['TOTAL POPULATION']):
        try:
            # Remove commas and convert to int
            pop_value = str(row['TOTAL POPULATION']).replace(',', '')
            pop_text = f"<br><strong>Population:</strong> {int(float(pop_value)):,}"
        except:
            pop_text = ""
    else:
        pop_text = ""
    
    # Create popup
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 250px;">
        <div style="font-size: 16px; font-weight: bold; color: #1a1a1a; margin-bottom: 8px;">
            {row['DEVELOPMEN']}
        </div>
        <div style="font-size: 14px; color: #555; line-height: 1.6;">
            <strong>Borough:</strong> {row['BOROUGH']}<br>
            <strong>Built:</strong> <span style="font-size: 18px; color: #2c5aa0; font-weight: bold;">{int(row['COMPLETION_YEAR'])}</span><br>
            <strong>Era:</strong> {row['DECADE']}
            {pop_text}
        </div>
    </div>
    """
    
    # Add circle marker
    folium.CircleMarker(
        location=[centroid.y, centroid.x],
        radius=8,
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{row['DEVELOPMEN']} ({int(row['COMPLETION_YEAR'])})",
        color='black',
        weight=1.5,
        fillColor=decade_colors[row['DECADE']],
        fillOpacity=0.85
    ).add_to(m)

# Create custom legend
legend_html = '''
<div style="position: fixed; 
            top: 120px; 
            right: 20px; 
            background-color: white; 
            z-index: 9999; 
            padding: 15px;
            border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.2);
            font-size: 14px;
            line-height: 24px;">
    <div style="font-weight: bold; font-size: 16px; margin-bottom: 10px; color: #1a1a1a;">
        Construction Era
    </div>
'''

for decade in ['Pre-1940', '1940s', '1950s', '1960s', '1970s', '1980s', '1990s', '2000s+']:
    count = decade_counts.get(decade, 0)
    if count > 0:
        color = decade_colors[decade]
        legend_html += f'''
        <div style="display: flex; align-items: center; margin: 5px 0;">
            <div style="width: 24px; 
                        height: 24px; 
                        background-color: {color}; 
                        border: 2px solid black; 
                        margin-right: 10px;
                        border-radius: 3px;">
            </div>
            <span>{decade} ({count})</span>
        </div>
        '''

legend_html += '''
    <div style="margin-top: 10px; 
                padding-top: 10px; 
                border-top: 1px solid #ddd; 
                font-size: 12px; 
                color: #666;">
        Click developments for details
    </div>
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

# Save map
output_file = 'nycha_interactive_map.html'
m.save(output_file)


Could not load borough boundaries: HTTP Error 404: Not Found
Continuing without borough boundaries...
